# simv2 movement analysis on Google Colab

This notebook runs the Java/WALA movement analysis in a Colab VM. Use **Runtime > Change runtime type > High-RAM** before running it. GPU is not required because the current analysis is CPU/RAM bound.

## 1. Install Java 21

Colab runtimes are disposable, so rerun this setup after a runtime restart.

In [ ]:
%%bash
set -euxo pipefail

apt-get update -qq
if apt-cache show openjdk-21-jdk-headless >/dev/null 2>&1; then
  apt-get install -y openjdk-21-jdk-headless
else
  apt-get install -y wget gpg ca-certificates
  install -d -m 0755 /etc/apt/keyrings
  wget -qO- https://packages.adoptium.net/artifactory/api/gpg/key/public \
    | gpg --dearmor > /etc/apt/keyrings/adoptium.gpg
  . /etc/os-release
  echo "deb [signed-by=/etc/apt/keyrings/adoptium.gpg] https://packages.adoptium.net/artifactory/deb ${VERSION_CODENAME} main" \
    > /etc/apt/sources.list.d/adoptium.list
  apt-get update -qq
  apt-get install -y temurin-21-jdk
fi

java -version
javac -version


## 1b. Tune the runtime VM (huge pages + memory mappings)

Rerun after a runtime restart. Two host-level knobs that matter for the slice:

- **`vm.max_map_count`** — ZGC multi-maps the heap; at a ~225 GB max heap it needs ~412k mappings but the default is 65530. Too low can throw a *premature* `OutOfMemoryError` mid-slice (the JVM warns about this at startup). Raise it.
- **Transparent huge pages** — the backward slice is a single-threaded, pointer-chasing IFDS solver over a multi-GB live set. With 4 KB pages the TLB covers only a few MB, so almost every access triggers a page-table walk. Enabling THP (2 MB pages) widens TLB reach ~512× and is the main lever for single-core throughput here.

These are OS commands, not JVM flags — they run in the shell, no Gradle involved.

In [ ]:
import subprocess
from pathlib import Path

# Headroom for ZGC's heap multi-mappings (default 65530 is far too low for a
# ~225 GB heap). Idempotent; takes effect immediately for new processes.
subprocess.run(["sysctl", "-w", "vm.max_map_count=1048576"], check=True)

# Enable transparent huge pages so -XX:+UseTransparentHugePages (set via
# -PanalysisLargePages) can back the heap with 2 MB pages. 'always' lets the
# kernel hand them out without the app asking; 'madvise' defrag keeps page
# compaction from stalling unrelated allocations.
def write_sys(path, value):
    p = Path(path)
    if not p.exists():
        print(f"skip (not present): {path}")
        return
    p.write_text(value + "\n")
    print(f"{path} -> {p.read_text().strip()}")

write_sys("/sys/kernel/mm/transparent_hugepage/enabled", "always")
write_sys("/sys/kernel/mm/transparent_hugepage/defrag", "madvise")

print("vm.max_map_count:", subprocess.run(
    ["sysctl", "-n", "vm.max_map_count"], capture_output=True, text=True).stdout.strip())


## 2. Clone the repository

Push your local changes to a branch first if the analysis code in Colab needs to match your workstation exactly.

In [ ]:
import os
import shlex
import subprocess
import sys
import shutil
from pathlib import Path

def run_checked(cmd, cwd=None, log_path=None):
    """Run a command, streaming its output live while teeing it to a log file.

    subprocess.run(..., PIPE) buffers everything and prints only on exit, so a
    long task (e.g. :analysis:runWala) shows a blank cell until it finishes.
    Here we read stdout line by line and flush, so Colab shows progress live.
    """
    print("$", " ".join(shlex.quote(str(part)) for part in cmd), flush=True)
    log_file = open(log_path, "w", encoding="utf-8") if log_path else None
    proc = subprocess.Popen(
        cmd,
        cwd=cwd,
        text=True,
        bufsize=1,  # line-buffered
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    try:
        for line in proc.stdout:
            sys.stdout.write(line)
            sys.stdout.flush()
            if log_file:
                log_file.write(line)
                log_file.flush()
    finally:
        proc.stdout.close()
        returncode = proc.wait()
        if log_file:
            log_file.close()
    if log_path:
        print("Log:", log_path, flush=True)
    if returncode != 0:
        raise subprocess.CalledProcessError(returncode, cmd)
    return returncode

REPO_URL = "https://github.com/Murat65536/simv2.git"
BRANCH = "analysis"  # Must contain the WALA analysis Gradle task and memory flags.
REPO_DIR = "/content/simv2"

# Re-running this cell removes REPO_DIR. If a previous run left the process cwd
# inside it, deleting it leaves the kernel with no valid cwd and the next
# subprocess fails with "Unable to read current working directory". chdir to a
# stable parent first so re-runs are idempotent.
os.chdir("/content")
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

clone_cmd = ["git", "clone"]
if BRANCH:
    clone_cmd += ["--branch", BRANCH, "--single-branch"]
clone_cmd += [REPO_URL, REPO_DIR]
run_checked(clone_cmd, log_path="/content/simv2-git-clone.log")

os.chdir(REPO_DIR)
print("Repo:", os.getcwd())
run_checked(["git", "branch", "--show-current"])
run_checked(["git", "rev-parse", "--short", "HEAD"])


## 3. Configure memory

`gradle.properties` in the repo is tuned for a large local machine. This cell keeps Gradle's own heap small and configures the analysis JVM without fixed heap/metaspace caps.

In [ ]:
from pathlib import Path

def total_ram_gb():
    with open("/proc/meminfo", "r", encoding="utf-8") as meminfo:
        for line in meminfo:
            if line.startswith("MemTotal:"):
                return int(line.split()[1]) / 1024 / 1024
    raise RuntimeError("Could not read total memory")

import os

TOTAL_RAM_GB = total_ram_gb()
CPU_COUNT = os.cpu_count()
GRADLE_DAEMON_XMX_GB = 4
ANALYSIS_XMX = "unlimited"
ANALYSIS_MAX_METASPACE = "unlimited"
ANALYSIS_MAX_RAM_PERCENTAGE = 90
# The slice is single-threaded; the many cores only feed GC. ParallelGC's
# full-GC pause scales with heap size and stalls the slice on a huge heap, so
# on Colab use a concurrent collector: 'zgc' (pause independent of heap) or
# 'g1'. Keep 'parallel' only to reproduce the local small-heap behavior.
ANALYSIS_GC = "zgc"
# Stream GC telemetry (Pause/heap lines) so a GC-bound run is visible live.
ANALYSIS_GC_LOG = True
# Back the heap with transparent huge pages to cut TLB misses on the slice's
# multi-GB pointer-chasing working set. Requires the THP setup cell to have run
# (/sys/kernel/mm/transparent_hugepage/enabled = always).
ANALYSIS_LARGE_PAGES = True

gradle_home = Path.home() / ".gradle"
gradle_home.mkdir(exist_ok=True)
gradle_props = gradle_home / "gradle.properties"
gradle_props.write_text(
    f"org.gradle.jvmargs=-Xmx{GRADLE_DAEMON_XMX_GB}G -Dfile.encoding=UTF-8\n"
    "org.gradle.daemon=false\n"
    "org.gradle.parallel=true\n",
    encoding="utf-8",
)

print(f"Total VM RAM: {TOTAL_RAM_GB:.1f} GB")
print(f"CPU count:    {CPU_COUNT}")
print(f"Gradle heap:  {GRADLE_DAEMON_XMX_GB} GB")
print(f"Analysis fixed heap cap: {ANALYSIS_XMX}")
print(f"Analysis fixed metaspace cap: {ANALYSIS_MAX_METASPACE}")
print(f"Analysis MaxRAMPercentage: {ANALYSIS_MAX_RAM_PERCENTAGE}")
print(f"Analysis GC: {ANALYSIS_GC} (gc log: {ANALYSIS_GC_LOG}, large pages: {ANALYSIS_LARGE_PAGES})")
print(gradle_props.read_text(encoding="utf-8"))


## 4. Warm the Fabric Loom cache

The analysis task needs Loom's merged Minecraft jar. Compiling the root mod first causes Loom to download and prepare the Minecraft dependencies.

In [ ]:
run_checked(["chmod", "+x", "./gradlew"])
run_checked(
    ["./gradlew", "--no-daemon", "--console=plain", ":compileJava", "--stacktrace"],
    log_path="/content/simv2-compileJava.log",
)


In [ ]:
from pathlib import Path

roots = [Path(".gradle"), Path.home() / ".gradle"]
merged_jars = []
for root in roots:
    if root.exists():
        merged_jars.extend(root.rglob("minecraft-merged-*.jar"))

print(f"Found {len(merged_jars)} merged Minecraft jar candidates")
for path in merged_jars[:20]:
    print(path)


## 5. Run the movement analysis

Leave `MC_JAR` empty for Gradle auto-detection. Set it only if the previous cell found a jar but `:analysis:runWala` cannot resolve it.

In [ ]:
MC_JAR = ""       # Optional explicit path to minecraft-merged-*.jar
SOURCES_JAR = ""  # Optional explicit sources jar path, or "-" for none
OUTPUT_DIR = "/content/simv2/src/main/generated"

cmd = [
    "./gradlew",
    "--no-daemon",
    "--console=plain",
    ":analysis:runWala",
    f"-PanalysisXmx={ANALYSIS_XMX}",
    f"-PanalysisMaxMetaspace={ANALYSIS_MAX_METASPACE}",
    f"-PanalysisMaxRamPercentage={ANALYSIS_MAX_RAM_PERCENTAGE}",
    f"-PanalysisGc={ANALYSIS_GC}",
    f"-PanalysisGcLog={'true' if ANALYSIS_GC_LOG else 'false'}",
    f"-PanalysisLargePages={'true' if ANALYSIS_LARGE_PAGES else 'false'}",
    f"-PoutputDir={OUTPUT_DIR}",
    "--stacktrace",
]
if MC_JAR:
    cmd.append(f"-PmcJar={MC_JAR}")
if SOURCES_JAR:
    cmd.append(f"-PsourcesJar={SOURCES_JAR}")

# JVM flags (-XX/-Xlog/-Xmx) for the analysis process are injected via these -P
# properties into analysisJvmArgs in build.gradle, NOT placed on the gradlew
# command line (Gradle would reject -X... as an unknown CLI option).
# The analysis prints a [slice-progress] heartbeat; run_checked streams it live.
run_checked(cmd, log_path="/content/simv2-runWala.log")


## 6. Download artifacts

The VM is temporary. Download the result zip or copy it to Google Drive before disconnecting the runtime.

In [ ]:
from pathlib import Path
import shutil
from google.colab import files

artifact_dir = Path(OUTPUT_DIR)
for path in sorted(artifact_dir.iterdir()):
    print(path.name, path.stat().st_size, "bytes")

zip_base = "/content/simv2-movement-artifacts"
zip_path = shutil.make_archive(zip_base, "zip", artifact_dir)
files.download(zip_path)


## Optional: copy artifacts to Google Drive

In [ ]:
# from google.colab import drive
# import shutil
# from pathlib import Path
#
# drive.mount("/content/drive")
# drive_dir = Path("/content/drive/MyDrive/simv2-analysis")
# drive_dir.mkdir(parents=True, exist_ok=True)
# for path in Path(OUTPUT_DIR).iterdir():
#     shutil.copy2(path, drive_dir / path.name)
# print("Copied artifacts to", drive_dir)
